# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Matías Taborda
- Alumno 2 :    -

In [2]:
import torch
# import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.datasets import fetch_lfw_people
# from sklearn.manifold import TSNE
# from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image
# import os
# import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__} | Dispositivo: {device}")

PyTorch: 2.11.0+cpu | Dispositivo: cpu


In [3]:
from PIL import Image
import torchvision.transforms as T

# Cargo el dataset LFW como base
lfw = fetch_lfw_people(min_faces_per_person=70, color=True, resize=1.0)

X_lfw = lfw.images    # (n_samples, 125, 94, 3) float en [0, 1]
y_lfw = lfw.target     # etiquetas numéricas
NOMBRES_LFW = lfw.target_names

print(f"Total de imágenes: {X_lfw.shape[0]}")
print(f"Tamaño de imagen: {X_lfw.shape[1:3]}")
print(f"Rango de valores: [{X_lfw.min():.2f}, {X_lfw.max():.2f}]")
print(f"\nPersonas ({len(NOMBRES_LFW)}):")

from collections import Counter
conteo = Counter(y_lfw)
for idx, nombre in enumerate(NOMBRES_LFW):
    print(f"  {nombre}: {conteo[idx]} imágenes")


# Convierto imágenes a PIL

X_lfw_pil = [Image.fromarray((img * 255).astype(np.uint8)) for img in X_lfw]

Total de imágenes: 1288
Tamaño de imagen: (125, 94)
Rango de valores: [0.00, 1.00]

Personas (7):
  Ariel Sharon: 77 imágenes
  Colin Powell: 236 imágenes
  Donald Rumsfeld: 121 imágenes
  George W Bush: 530 imágenes
  Gerhard Schroeder: 109 imágenes
  Hugo Chavez: 71 imágenes
  Tony Blair: 144 imágenes


In [4]:
# Verifico mi dataset 
from pathlib import Path
dataset_path = Path("src/data/dataset_tp1")
dataset_folders = []
 
for folder in dataset_path.iterdir():
    if folder.is_dir():
        dataset_folders.append(folder)

# verifico imágenes en las carpetas
print("*"*20)
print("Dataset TP1")
print("*"*20)
for folder in dataset_folders:
    print(f"{folder.name}: {len(list(folder.iterdir()))} imágenes")

********************
Dataset TP1
********************
gatacia: 2 imágenes
persona_1: 3 imágenes
persona_2: 4 imágenes


In [6]:
# Recorro las carpetas y cargo las imágenes 
X = []         
y = []
names = []

for persona_id, folder in enumerate(dataset_path.iterdir()):
    if folder.is_dir():
        names.append(folder.name)
        for img_file in folder.glob('*.jpg'):
            img = Image.open(img_file).convert('RGB')
            img = img.resize((94, 125)) # mismas dimensiones en LFW
            X.append(img) 
            y.append(persona_id) 
 

print(f"Dataset: {len(X)} imágenes, {len(names)} personas")

Dataset: 9 imágenes, 3 personas


In [7]:
if len(X) > 0:
    # datasets combinados
    X_combined = X_lfw_pil + X
    y_combined = list(y_lfw) + y
    combined_names = list(NOMBRES_LFW) + names
    print(f"Dataset obtenido: {len(X_combined)} imágenes | {len(combined_names)} personas")
else:
    print("Dataset vacío")

transform = T.Compose([
    T.ToTensor(),  # PIL -> Tensor
])

class FaceDataset(torch.utils.data.Dataset):
    def __init__(self, images_pil, labels, transform=None):
        self.images = images_pil
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

combined_dataset = FaceDataset(X_combined, y_combined, transform=transform)


Dataset obtenido: 1297 imágenes | 10 personas


In [8]:
# Divide con subset
train_size = int(0.8 * len(combined_dataset))
train_dataset = torch.utils.data.Subset(combined_dataset, range(train_size))
val_dataset = torch.utils.data.Subset(combined_dataset, range(train_size, len(combined_dataset)))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
 
print(f"Train: {len(train_dataset)} imágenes")
print(f"Val: {len(val_dataset)} imágenes")
print(f"Clases: {len(combined_names)}")

Train: 1037 imágenes
Val: 260 imágenes
Clases: 10


In [9]:
import torch.nn as nn
from torchvision.models import ResNet18_Weights

# Cargo ResNet18 pre-entrenado
modelo_ft = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
 
print("Cabeza original de ResNet18 (ImageNet):")
print(f"  avgpool: {modelo_ft.avgpool}")
print(f"  fc:      {modelo_ft.fc}   → 1000 clases ImageNet")

# Reemplazo la cabeza para combined_dataset (10 clases)
NUM_CLASES = 10
DIM_FEATURES = modelo_ft.fc.in_features   # 512 para ResNet18
modelo_ft.fc = nn.Linear(DIM_FEATURES, NUM_CLASES)
modelo_ft = modelo_ft.to(device)
 
print(f"\nNueva cabeza para combined_dataset:")
print(f"  fc: {modelo_ft.fc}   → {NUM_CLASES} clases")
 
# Parámetros totales vs entrenables
total_p    = sum(p.numel() for p in modelo_ft.parameters())
train_p    = sum(p.numel() for p in modelo_ft.parameters() if p.requires_grad)
print(f"\nParámetros totales:      {total_p:,}")
print(f"Parámetros entrenables:  {train_p:,}")

Cabeza original de ResNet18 (ImageNet):
  avgpool: AdaptiveAvgPool2d(output_size=(1, 1))
  fc:      Linear(in_features=512, out_features=1000, bias=True)   → 1000 clases ImageNet

Nueva cabeza para combined_dataset:
  fc: Linear(in_features=512, out_features=10, bias=True)   → 10 clases

Parámetros totales:      11,181,642
Parámetros entrenables:  11,181,642


In [10]:
def entrenar_epoca(modelo, loader, criterion, optimizer):
    modelo.train()
    total_loss, correctas, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = modelo(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correctas += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, 100 * correctas / total

def evaluar(modelo, loader, criterion):
    modelo.eval()
    total_loss, correctas, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = modelo(imgs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            correctas += (logits.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, 100 * correctas / total

criterion = nn.CrossEntropyLoss()

In [12]:
# Reiniciamos el modelo y congelamos el backbone
modelo_fe = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for param in modelo_fe.parameters():
    param.requires_grad = False

modelo_fe.fc = nn.Linear(DIM_FEATURES, NUM_CLASES) 
modelo_fe = modelo_fe.to(device)

n_train = sum(p.numel() for p in modelo_fe.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in modelo_fe.parameters())
print(f"Parámetros entrenables: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.1f}%)")

optimizer_fe = optim.Adam(filter(lambda p: p.requires_grad, modelo_fe.parameters()), lr=1e-3)

hist_fe = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
N_EPOCHS_FE = 10

print(f"\nEntrenando {N_EPOCHS_FE} épocas (Feature Extraction)...")
for ep in range(1, N_EPOCHS_FE + 1):
    tr_l, tr_a = entrenar_epoca(modelo_fe, train_loader, criterion, optimizer_fe)
    vl_l, vl_a = evaluar(modelo_fe, val_loader, criterion)
    hist_fe["train_loss"].append(tr_l); hist_fe["train_acc"].append(tr_a)
    hist_fe["val_loss"].append(vl_l);   hist_fe["val_acc"].append(vl_a)
    print(f"  Época {ep}/{N_EPOCHS_FE}  |  train loss {tr_l:.4f}  acc {tr_a:.1f}%  "
          f"|  val loss {vl_l:.4f}  acc {vl_a:.1f}%")


##########################################
# Resultado con 5 épocas:
'''
Parámetros entrenables: 5,130 / 11,181,642  (0.0%)

Entrenando 5 épocas (Feature Extraction)...
  Época 1/5  |  train loss 1.6173  acc 43.9%  |  val loss 1.4489  acc 53.1%
  Época 2/5  |  train loss 1.0141  acc 68.1%  |  val loss 1.0871  acc 69.2%
  Época 3/5  |  train loss 0.8124  acc 75.1%  |  val loss 1.0059  acc 68.8%
  Época 4/5  |  train loss 0.6947  acc 79.3%  |  val loss 0.9686  acc 72.3%
  Época 5/5  |  train loss 0.6476  acc 78.0%  |  val loss 0.9176  acc 73.1%
'''

# Vemos que todavía hay margen de mejora, subimos las épocas a 10

Parámetros entrenables: 5,130 / 11,181,642  (0.0%)

Entrenando 10 épocas (Feature Extraction)...
  Época 1/10  |  train loss 1.6347  acc 45.0%  |  val loss 1.4529  acc 56.2%
  Época 2/10  |  train loss 1.0819  acc 65.6%  |  val loss 1.1405  acc 63.5%
  Época 3/10  |  train loss 0.8501  acc 73.6%  |  val loss 0.9964  acc 70.4%
  Época 4/10  |  train loss 0.7412  acc 79.0%  |  val loss 0.9401  acc 71.9%
  Época 5/10  |  train loss 0.6681  acc 78.9%  |  val loss 0.9150  acc 71.5%
  Época 6/10  |  train loss 0.5834  acc 81.5%  |  val loss 0.8572  acc 76.2%
  Época 7/10  |  train loss 0.5288  acc 85.1%  |  val loss 0.8416  acc 76.2%
  Época 8/10  |  train loss 0.4664  acc 87.8%  |  val loss 0.8314  acc 76.5%
  Época 9/10  |  train loss 0.4579  acc 87.3%  |  val loss 0.8080  acc 77.7%
  Época 10/10  |  train loss 0.4169  acc 88.6%  |  val loss 0.8074  acc 76.9%


'\nParámetros entrenables: 5,130 / 11,181,642  (0.0%)\n\nEntrenando 5 épocas (Feature Extraction)...\n  Época 1/5  |  train loss 1.6173  acc 43.9%  |  val loss 1.4489  acc 53.1%\n  Época 2/5  |  train loss 1.0141  acc 68.1%  |  val loss 1.0871  acc 69.2%\n  Época 3/5  |  train loss 0.8124  acc 75.1%  |  val loss 1.0059  acc 68.8%\n  Época 4/5  |  train loss 0.6947  acc 79.3%  |  val loss 0.9686  acc 72.3%\n  Época 5/5  |  train loss 0.6476  acc 78.0%  |  val loss 0.9176  acc 73.1%\n'